In [13]:
import os
import numpy as np
import rasterio
from rasterio.features import shapes
from osgeo import gdal, ogr, osr

def raster_to_contours(raster_path, output_shapefile, contour_interval=0.5):
    # Open raster file
    src_ds = gdal.Open(raster_path)
    band = src_ds.GetRasterBand(1)  # First band

    # Create output shapefile
    driver = ogr.GetDriverByName("ESRI Shapefile")
    if os.path.exists(output_shapefile):
        driver.DeleteDataSource(output_shapefile)
    out_ds = driver.CreateDataSource(output_shapefile)
    
    # Define spatial reference system (same as raster)
    srs = osr.SpatialReference()
    srs.ImportFromWkt(src_ds.GetProjection())

    # Create layer
    out_layer = out_ds.CreateLayer("contours", srs, ogr.wkbLineString)
    field = ogr.FieldDefn("elevation", ogr.OFTReal)
    out_layer.CreateField(field)

    # Generate contour lines
    gdal.ContourGenerate(band, contour_interval, 0, [], 0, 0, out_layer, -1, 0)


    # Clean up
    src_ds = None
    out_ds = None
    print(f"Contour lines saved to {output_shapefile}")

# Example usage
raster_path = r"D:\Phd Research\Port_Lavaca_Wave_Model\Raster\idw_raster_water.tif"  # Replace with your raster file
output_shapefile = r"D:\Phd Research\Port_Lavaca_Wave_Model\contour3.shp"  # Output shapefile
contour_interval = 3.0  # Contour interval in meters

raster_to_contours(raster_path, output_shapefile, contour_interval)


Contour lines saved to D:\Phd Research\Port_Lavaca_Wave_Model\contour3.shp


In [16]:
import os
from osgeo import gdal, ogr, osr

def raster_to_contours(raster_path, output_shapefile, contour_interval=3.0):
    # Open raster
    src_ds = gdal.Open(raster_path)
    if src_ds is None:
        raise RuntimeError(f"Could not open raster file: {raster_path}")

    band = src_ds.GetRasterBand(1)
    no_data_value = band.GetNoDataValue() or -9999  # Ensure valid NoData value

    # Set projection (UTM WGS 1984 Zone 14N)
    srs = osr.SpatialReference()
    srs.ImportFromEPSG(32614)

    # Remove old shapefile if exists
    driver = ogr.GetDriverByName("ESRI Shapefile")
    if os.path.exists(output_shapefile):
        driver.DeleteDataSource(output_shapefile)

    # Create output shapefile
    out_ds = driver.CreateDataSource(output_shapefile)
    out_layer = out_ds.CreateLayer("contours", srs, ogr.wkbLineString)

    # Define attribute field
    field_def = ogr.FieldDefn("elevation", ogr.OFTReal)
    out_layer.CreateField(field_def)

    # Generate contours with elevation attribute
    gdal.ContourGenerate(band, contour_interval, 0, [], 1, no_data_value, out_layer, -1, 0)

    # Save and close
    out_ds = None
    print(f"Contours saved to {output_shapefile}")

# Example usage
raster_path = r"D:\Phd Research\Port_Lavaca_Wave_Model\Raster\idw_raster_water.tif"  # Replace with your raster file
output_shapefile = r"D:\Phd Research\Port_Lavaca_Wave_Model\contour6.shp"  # Output shapefile
contour_interval = 3.0  # Adjust as needed

raster_to_contours(raster_path, output_shapefile, contour_interval)


Contours saved to D:\Phd Research\Port_Lavaca_Wave_Model\contour6.shp
